# Test XGBoost Model với dữ liệu tháng 1/2025

**Mục đích:**
- Load model XGBoost đã train với data 2024
- Test với dữ liệu tháng 1/2025 (unseen data)
- Đánh giá performance

**Quy trình:**
1. Upload model đã train
2. Generate test data
3. Feature engineering
4. Predict và evaluate

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")
print("📅 Target: January 2025 weather data")

## Bước 1: Upload Model

Upload file `xgboost_health_classifier.pkl` từ máy tính

In [ ]:
from google.colab import files

print("📤 Uploading model...")
uploaded = files.upload()

model_filename = list(uploaded.keys())[0]
with open(model_filename, 'rb') as f:
    model_data = pickle.load(f)

model = model_data['model']
label_encoder = model_data['label_encoder']
feature_columns = model_data['feature_columns']

print(f"✅ Model loaded: {len(feature_columns)} features")
print(f"Classes: {label_encoder.classes_}")

## Bước 2: Generate Test Data (Jan 2025)

In [ ]:
TEST_CITIES = [
    {'name': 'Ho Chi Minh City', 'id': 1580578},
    {'name': 'Hanoi', 'id': 1581130},
    {'name': 'Da Nang', 'id': 1583992},
    {'name': 'Can Tho', 'id': 1581188},
    {'name': 'Hai Phong', 'id': 1581349}
]

START_DATE = datetime(2025, 1, 1)
np.random.seed(42)

data = []
for city in TEST_CITIES:
    for day in range(31):
        date = START_DATE + timedelta(days=day)
        base_temp = 22 if city['name'] == 'Ho Chi Minh City' else 18
        temp = np.random.normal(base_temp, 4)
        data.append({
            'city_id': city['id'],
            'city_name': city['name'],
            'date': date.strftime('%Y-%m-%d'),
            'temperature': temp,
            'temp_min': temp - np.random.uniform(2, 5),
            'temp_max': temp + np.random.uniform(3, 7),
            'humidity': np.random.normal(75, 12),
            'precipitation': max(0, np.random.exponential(2)),
            'wind_speed': max(0, np.random.normal(3, 1.5)),
            'uv_index': max(0, np.random.normal(5, 2)),
            'pm2_5': max(0, np.random.lognormal(3, 0.8)),
            'pm10': max(0, np.random.lognormal(3.5, 0.8)),
            'aqi': np.random.choice([1,2,3,4,5], p=[0.15,0.3,0.35,0.15,0.05]),
            'co': max(0, np.random.normal(300, 100)),
            'no2': max(0, np.random.normal(20, 10)),
            'o3': max(0, np.random.normal(50, 20)),
            'so2': max(0, np.random.normal(5, 3)),
            'nh3': max(0, np.random.normal(2, 1))
        })

df = pd.DataFrame(data)
print(f"✅ Generated {len(df)} test samples")
print(df.head())

## Bước 3: Feature Engineering

In [ ]:
# Create derived features
df['temp_range'] = df['temp_max'] - df['temp_min']
df['is_hot'] = (df['temperature'] > 32).astype(int)
df['is_cold'] = (df['temperature'] < 18).astype(int)
df['is_humid'] = (df['humidity'] > 70).astype(int)
df['is_dry'] = (df['humidity'] < 40).astype(int)
df['uv_high'] = (df['uv_index'] > 6).astype(int)
df['has_rain'] = (df['precipitation'] > 5).astype(int)
df['pm25_high'] = (df['pm2_5'] > 25).astype(int)
df['aqi_poor'] = (df['aqi'] >= 4).astype(int)

print("✅ Features created")

## Bước 4: Calculate Ground Truth

In [ ]:
def calculate_risk_score(row):
    score = 0
    if row['temperature'] < 10 or row['temperature'] > 35: score += 3
    elif row['temperature'] < 15 or row['temperature'] > 32: score += 2
    elif row['temperature'] < 18 or row['temperature'] > 27: score += 1
    if row['humidity'] < 30 or row['humidity'] > 70: score += 2
    elif row['humidity'] < 40 or row['humidity'] > 60: score += 1
    uv = row['uv_index']
    if uv > 11: score += 3
    elif uv > 8: score += 2
    elif uv > 6: score += 1
    pm25 = row['pm2_5']
    if pm25 > 55.5: score += 4
    elif pm25 > 37.5: score += 3
    elif pm25 > 25: score += 2
    elif pm25 > 15: score += 1
    if row['aqi'] >= 5: score += 2
    elif row['aqi'] >= 4: score += 1
    return min(score, 15)

def assign_risk(score):
    if score <= 2: return 'safe'
    elif score <= 5: return 'low'
    elif score <= 8: return 'moderate'
    elif score <= 12: return 'high'
    else: return 'extreme'

df['risk_score'] = df.apply(calculate_risk_score, axis=1)
df['true_risk'] = df['risk_score'].apply(assign_risk)
print(df['true_risk'].value_counts())

## Bước 5: Predict

In [ ]:
X_test = df[feature_columns].fillna(0)
y_pred_encoded = model.predict(X_test)
y_pred = label_encoder.inverse_transform(y_pred_encoded)
df['predicted_risk'] = y_pred

print("✅ Predictions completed")
print(df[['date', 'city_name', 'temperature', 'pm2_5', 'true_risk', 'predicted_risk']].head(10))

## Bước 6: Evaluate Performance

In [ ]:
y_true = label_encoder.transform(df['true_risk'])
accuracy = accuracy_score(y_true, y_pred_encoded)

print("📊 MODEL PERFORMANCE")
print("=" * 60)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Correct: {(y_true == y_pred_encoded).sum()} / {len(y_true)}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred_encoded, target_names=label_encoder.classes_, digits=3))

## Bước 7: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred_encoded)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - January 2025 Test')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## Bước 8: Error Analysis

In [ ]:
df['correct'] = (df['true_risk'] == df['predicted_risk'])
errors = df[~df['correct']]

print(f"🔍 ERROR ANALYSIS")
print(f"Errors: {len(errors)} / {len(df)} ({len(errors)/len(df)*100:.2f}%)")

if len(errors) > 0:
    print("\nTop 10 errors:")
    print(errors[['date', 'city_name', 'temperature', 'pm2_5', 'true_risk', 'predicted_risk']].head(10))
else:
    print("\n🎉 Perfect predictions!")

## ✅ Tổng kết

**Kết quả:**
- Test samples: 155 (5 cities × 31 days)
- Model: XGBoost trained on 2024 data
- Features: 23 (weather + UV + air quality + derived)
- Ground truth: WHO thresholds

**Performance:**
- Accuracy: [see above]
- Precision/Recall: [see classification report]
- Errors: [see error analysis]

**Next steps:**
- If accuracy < 80%: retrain with more data
- If class imbalance: apply SMOTE
- If errors common: add interaction features